# News2Stock Anaylist LoRA Finetuning

In [1]:
# Runpod에 설치 되지 않은 패키지 설치
# accelerate: 멀티 gpu 환경 분산 학습 및 최적화
# trl: sft(지도 미세 조정)을 위한 Trainer 클래스 및 설정 클래스 제공
# peft: 다양한 peft 기법 지원(LoRA)
%pip install transformers datasets accelerate peft trl hf_transfer pydantic langchain-huggingface

   ---------------------------------------- 0.0/680.7 kB ? eta -:--:--
   ---------------------------------------- 680.7/680.7 kB 8.9 MB/s  0:00:00
   ---------------------------------------- 0.0/751.0 kB ? eta -:--:--
   ---------------------------------------- 751.0/751.0 kB 10.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 11.5 MB/s  0:00:00

   ------------- -------------------------- 1/3 [trl]
   ------------- -------------------------- 1/3 [trl]
   ------------- -------------------------- 1/3 [trl]
   ------------- -------------------------- 1/3 [trl]
   ------------- -------------------------- 1/3 [trl]
   ------------- -------------------------- 1/3 [trl]
   ------------- -------------------------- 1/3 [trl]
   ------------- -------------------------- 1/3 [trl]
   ------------- -------------------------- 1/3 [trl]
   ------------- -------------------------- 1/3 [trl]
   ------------- --

In [2]:
# 로컬 (.env 파일에서 가져옴)
from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.environ['HF_TOKEN']

In [ ]:
# Runpod(서버 환경변수에서 가져옴)
import os
HF_TOKEN = os.environ['HF_TOKEN']

In [4]:
# 데이터셋 로드
from datasets import load_dataset 

dataset = load_dataset('blimu/naver-economy-news2stock2', split='train')
print(len(dataset))
dataset

1000


Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 1000
})

In [5]:
dataset[0]

{'system': "\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,\n이유/근거 등을 분석하는 금융 분석 전문가입니다.\n\n다음 출력 지시사항을 지켜주세요.\n1. 뉴스와 종목간의 영향성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 영향성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정 영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n",
 'user': '추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크

## 데이터셋 분할

In [6]:
test_ratio = 0.2

train_data = []
test_data = []

data_indices = list(range(len(dataset)))
test_size = int(len(dataset) * test_ratio)

train_data_indices = data_indices[test_size:]
test_data_indices = data_indices[:test_size]

# 학습/평가 데이터셋 형식 지정 함수
def format_data(data):
    return {
        'messages' : [
            {'role' : 'system', 'content': data['system']},
            {'role' : 'user', 'content': data['user']},
            {'role' : 'assistant', 'content': data['assistant']},
        ]
    }

train_data = [format_data(dataset[i]) for i in train_data_indices]
test_data = [format_data(dataset[i]) for i in test_data_indices]

print('학습셋 : ', len(train_data))
print('평가셋 : ', len(test_data))

학습셋 :  800
평가셋 :  200


In [7]:
# 값 하나 확인
train_data[128]

{'messages': [{'role': 'system',
   'content': "\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,\n이유/근거 등을 분석하는 금융 분석 전문가입니다.\n\n다음 출력 지시사항을 지켜주세요.\n1. 뉴스와 종목간의 영향성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 영향성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정 영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n"},
  {'role': 'user',
   'content': '에어부산 울란바토르·오사카 노선 재개\n에어부산이 김해국제공항에서 출발하는 몽골 울란바토르와 일본 오사카 노선 운항을 각각 주 2회 일정으로 코로나19 팬데믹 사태 이후 28개월 만에 재개한다고 1일 밝혔다. 부산 울란바토르 노선은 김해국제공항에서 오전 8시 25분에 출발해 현지 공항에 오전 11시 40분 도착하고 귀국편은 오후 1시에 출발해 김해공항에 오후 5시 30분 도착하는 일정으로 주 2회 운항한다. 몽골은 입국 시 코로나19 검사와 백신 접종 여부를 확인하지 않아 자유롭게 여행이 가능한 국가다. 부산 오사카 노선은 김해공항에서 오전 8시 35분에 출발해 간사이공항에 오전 10시 도착 귀국편은 간사이공항에서 낮 12시에 출발해 김해공항에 오후 1시 30분 도착하는 일정으로 주 2회 운항

In [8]:
# DataSet 객체로 다시 변환
from datasets import Dataset 

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

In [ ]:
# 값 하나 확인 -> 내용 변환 없이 타입 변환만 수행
train_dataset[128]

{'messages': [{'role': 'system',
   'content': "\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,\n이유/근거 등을 분석하는 금융 분석 전문가입니다.\n\n다음 출력 지시사항을 지켜주세요.\n1. 뉴스와 종목간의 영향성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 영향성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정 영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n"},
  {'role': 'user',
   'content': '에어부산 울란바토르·오사카 노선 재개\n에어부산이 김해국제공항에서 출발하는 몽골 울란바토르와 일본 오사카 노선 운항을 각각 주 2회 일정으로 코로나19 팬데믹 사태 이후 28개월 만에 재개한다고 1일 밝혔다. 부산 울란바토르 노선은 김해국제공항에서 오전 8시 25분에 출발해 현지 공항에 오전 11시 40분 도착하고 귀국편은 오후 1시에 출발해 김해공항에 오후 5시 30분 도착하는 일정으로 주 2회 운항한다. 몽골은 입국 시 코로나19 검사와 백신 접종 여부를 확인하지 않아 자유롭게 여행이 가능한 국가다. 부산 오사카 노선은 김해공항에서 오전 8시 35분에 출발해 간사이공항에 오전 10시 도착 귀국편은 간사이공항에서 낮 12시에 출발해 김해공항에 오후 1시 30분 도착하는 일정으로 주 2회 운항

# NCSOFT/Llama-VARCO-8B-Instruct란?

https://huggingface.co/NCSOFT/Llama-VARCO-8B-Instruct

- 기본 모델: Meta의 Llama-3.1-8B 모델을 기반으로 한다.
- 개발 목적: 한국어 능력을 극대화하는 동시에 영어 구사 능력도 유지하도록 설계되었다.
- 학습 방법: 한국어와 영어 데이터셋을 활용한 지속 사전 학습(Continual Pre-training)을 거쳤으며, 이후 지도 미세 조정(SFT)과 직접 선호도 최적화(DPO)를 통해 인간의 선호도에 맞게 정렬되었다.

## SFT에서 한국어능력향상과 동시에 영어능력유지란?

일반적으로 한국어 데이터를 대량으로 추가 학습시키면 기존에 모델이 가지고 있던 영어 지식이 손상되는 파괴적 망각(Catastrophic Forgetting) 현상이 발생한다. 엔씨소프트는 이를 방지하기 위해 지속 사전 학습(Continual Pre-training)을 적용했다.

### 1. 데이터 믹스(Data Mixing) 전략

단순히 한국어 데이터만 밀어 넣는 것이 아니라, 모델이 이미 학습했던 영어 데이터와 고품질의 한국어 데이터를 특정 비율로 섞어 학습한다. 이를 통해 기존의 영어 추론 능력을 잃지 않으면서 새로운 언어 체계를 습득하게 된다.

### 2. 토크나이저 효율화와 임베딩 확장

기존 Llama-3.1의 토크나이저 성능을 유지하면서 한국어 표현력을 높이기 위해 어휘 사전(Vocabulary)을 최적화한다. 영어 토큰 정보는 건드리지 않고 한국어 토큰의 밀도를 높여 두 언어 간의 연결 고리를 강화하는 방식이다.

### 3. 지식 전이(Knowledge Transfer)

영어 데이터로 학습된 모델의 강력한 논리적 사고 능력을 한국어로 전이시키는 과정을 거친다.

- 추론 능력 유지: 수학이나 코딩 같은 논리적 작업은 영어 데이터에서 배운 구조를 그대로 활용한다.
- 언어 정렬: SFT(지도 미세 조정) 단계에서 동일한 질문을 한국어와 영어로 번갈아 학습시켜, 언어에 상관없이 일관된 답변을 내놓도록 유도한다.

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = 'NCSOFT/Llama-VARCO-8B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--NCSOFT--Llama-VARCO-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


# llama-3 chat template 변환

Llama3 모델은 특정 chat template 형식으로 학습되어, 그 형식을 사용해야 최적 성능을 낼 수 있다. Chat template을 사용하지 않으면 모델이 대화 구조를 제대로 인식하지 못할 수 있다. open_ai 형식의 데이터를 llama-3 형식으로 변환한다.

LLaMA-3 채팅 포맷

LLaMA-3 채팅 포맷은 LLaMA-3 계열 챗봇 모델이 대화 내용을 이해하고 답변할 수 있도록 만들어진 입력 데이터 구조이다. 여러 역할(시스템, 유저, 어시스턴트)의 메시지를 특별한 토큰과 구조로 묶어서 하나의 프롬프트로 합치는 방식이다. 구조 예시는 아래와 같이 대화 흐름을 명확히 구분하는 토큰들이 사용된다.

~~~text
<|begin_of_text|>

<|start_header_id|>system<|end_header_id|>
[시스템 역할 지침]<|eot_id|>

<|start_header_id|>user<|end_header_id|>
[유저 질문]<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
[모델의 답변]<|eot_id|>
~~~

- `<|begin_of_text|>`: 전체 프롬프트의 시작을 알리는 토큰
- `<|start_header_id|>role<|end_header_id|>`: 각 메시지의 역할 구분(시스템, 유저, 어시스턴트 등)
- 각 메시지 끝에 `<|eot_id|>`: 하나의 메시지 블록이 끝났음을 알림
- 마지막 assistant 블록은 응답 생성 위치를 가리킨다.

왜 이 포맷이 필요할까?

- 모델이 “어디까지가 시스템 안내, 어디서부터가 유저 질문, 어디서부터가 답변인지” 정확하게 파악할 수 있다.
- 여러 턴(turn)의 대화가 이어질 때도 메시지 경계를 명확히 구분해 혼동 없이 맥락을 유지할 수 있다.
- LLaMA-3 계열 모델은 이런 포맷으로 학습되어 있기 때문에 실전 파인튜닝/추론 시에도 반드시 이 구조로 입력해야 기대하는 챗봇 성능을 발휘할 수 있다.

In [ ]:
# apply_chat_template 함수
# openai 방식의 메세지를 llama3 방식으로 변환
text = tokenizer.apply_chat_template(train_dataset[128]['messages'], tokenize=False)
print(text)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,
이유/근거 등을 분석하는 금융 분석 전문가입니다.

다음 출력 지시사항을 지켜주세요.
1. 뉴스와 종목간의 영향성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 영향성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정 영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>

에어부산 울란바토르·오사카 노선 재개
에어부산이 김해국제공항에서 출발하는 몽골 울란바토르와 일본 오사카 노선 운항을 각각 주 2회 일정으로 코로나19 팬데믹 사태 이후 28개월 만에 재개한다고 1일 밝혔다. 부산 울란바토르 노선은 김해국제공항에서 오전 8시 25분에 출발해 현지 공항에 오전 11시 40분 도착하고 귀국편은 오후 1시에 출발해 김해공항에 오후 5시 30분 도착하는 일정으로 주 2회 운항한다. 몽골은 입국 시 코로나19 검사와 백신 접종 여부를 확인하지 않아 자유롭게 여행이 가능한 국가다. 부산 오사카 노선은 김해공항에서 오전 8시 35분에 출발해 간사이공항에 오전 10시 도착 귀국편은 간사이공항에서 낮 12시에 출발해 김해공항에 오후 1시 30분 도착하는 일정

## data_collator 함수

data_collator 함수는 학습 과정에서 여러 개의 샘플을 하나의 미니배치(batch)로 묶고, 모델이 바로 학습할 수 있는 형태로 변환하는 역할을 한다.

- 미니배치(batch) 데이터를 모델이 바로 학습할 수 있는 형태(토큰, 마스크, 정답)로 변환한다.
- 특히 아래와 같은 LLaMA-3 채팅 포맷을 사용할 때, “어디까지가 질문이고 어디서부터가 답변(assistant)인지”를 정확히 구분해서 모델이 정답(답변 부분)만 학습하도록 레이블을 지정한다.

### 1. 프롬프트 생성 (Prompt Construction)

입력받은 batch 데이터는 리스트 내에 여러 메시지(system, user, assistant)를 포함하는 사전(dict) 구조이다.

- Llama 3의 특수 토큰(`<|begin_of_text|>`, `<|start_header_id|>`, `<|eot_id|>`)을 사용하여 모든 대화 내용을 하나의 긴 문자열로 병합한다.
- 각 역할(role)의 시작과 끝을 명확히 구분하여 모델이 대화 맥락을 이해할 수 있도록 구성한다.

### 2. 토크나이즈 및 패딩 (Tokenization)

병합된 문자열 리스트를 `tokenizer`를 통해 숫자 ID(`input_ids`)로 변환한다.

- `padding=True`: 배치 내의 문장들 중 가장 긴 문장을 기준으로 길이를 맞춘다.
- `truncation=True`: `max_length`를 초과하는 데이터는 절단한다.
- `return_tensors="pt"`: PyTorch 텐서 형식으로 결과를 반환한다.

### 3. 레이블 생성 및 Loss Masking

모델이 사용자의 질문이 아닌 모델의 답변(assistant) 부분에 대해서만 학습하도록 설정한다.

- `-100` 값의 의미: PyTorch의 `CrossEntropyLoss`는 레이블 값이 `-100`인 경우 손실(Loss) 계산에서 제외한다. 이를 통해 모델은 질문 부분을 예측하려고 노력하지 않고, 답변 부분의 정확도에만 집중하게 된다.
- 구간 탐색: `assistant_tokens`를 기준으로 답변이 시작되는 위치를 찾고, `<|eot_id|>` 토큰이 나오는 지점까지의 인덱스를 추출한다.
- 값 복사: 해당 구간의 `labels`에만 실제 `input_ids` 값을 복사하여 넣는다.

In [ ]:
def data_collator(batch, tokenizer=tokenizer, max_length=8192):
    # 1. 프롬프트 생성
    prompts = []
    for example in batch:
        prompt = '<|begin_of_text|>'
        for msg in example['messages']:
            role = msg['role']
            content = msg['content'].strip()
            prompt += f"<|start_header_id|>{role}<|end_header_id|>\n{content}<|eot_id|>"
        prompts.append(prompt)
    # display(prompts)

    # 2. 토큰처리/패딩/텐서변환
    tokenized = tokenizer(
        prompts,
        truncation=True,
        max_length=max_length,
        padding=True, # 배치 내에서 가장 긴 텍스트 기준 패딩 처리
        return_tensors='pt'
    )
    input_ids = tokenized['input_ids']
    attention_mask = tokenized['attention_mask']
    # print(tokenized)
    # print(len(tokenized['input_ids'][0]))
    # print(len((tokenized['input_ids'][1])))
    # print(len((tokenized['attention_mask'][0])))
    # print(len((tokenized['attention_mask'][1])))

    # 3. 라벨 생성
    labels = torch.full_like(input_ids, fill_value=-100)
    # print(labels.shape)

    assistant_header = '<|start_header_id|>assistant<|end_header_id|>\n'
    assistant_token_id = tokenizer.encode(assistant_header, add_special_tokens=False)
    # print(assistant_token_id)
    eot_token = '<|eot_id|>'
    eot_token_id = tokenizer.encode(eot_token, add_special_tokens=False)
    # print(eot_token_id)

    for i, ids in enumerate(input_ids):
        ids_list = ids.tolist()

        # assistant 답변 위치 찾기
        start = None
        for idx in range(len(ids_list) - len(assistant_token_id) + 1):
            if ids_list[idx: idx + len(assistant_token_id)] == assistant_token_id:
                start = idx + len(assistant_token_id)
                break

        # 답변 끝 위치 찾기
        if start is not None:
            end = None
            for idx in range(start, len(ids_list) - len(eot_token_id) + 1):
                if  ids_list[idx: idx + len(eot_token_id)] == eot_token_id:
                    end = idx + len(eot_token_id)
                    break
        
        # print(f'{i}: {start} ~ {end}')

        # 정답 부분은 -100 이 아닌 실제 값으로 변환
        labels[i, start:end] = input_ids[i, start:end]

    return {
        'input_ids' : input_ids,
        'attention_mask' : attention_mask,
        'labels' : labels
    }

data_collator([train_dataset[0], train_dataset[1]])

## Causal Language Model 파인튜닝: input_ids와 labels 구조 이해

### 데이터 구조

Causal Language Model을 파인튜닝할 때 학습 데이터는 보통 `input_ids`와 `labels`로 구성된다.

~~~text
input_ids: [system_tokens..., user_tokens..., assistant_tokens...]   # 프롬프트 + 정답 전체 시퀀스
labels:    [-100, -100, ..., -100, assistant_tokens...]              # assistant 답변 구간만 학습 대상
~~~

| 항목 | 내용 |
|---|---|
| Input IDs | 프롬프트 + 정답 전체 시퀀스 |
| Labels | `-100`은 프롬프트 구간, 실제 토큰 값은 답변 구간 |
| 결과 | 모델은 입력을 다 보지만, 오직 답변 부분을 예측하는 과정에서만 학습이 일어남 |

### 질문에 해당하는 input_ids에 이미 답이 포함되어 있다?

처음 보면 이런 의문이 생길 수 있다.

"답이 이미 있는데 어떻게 학습하는가?"

모델은 정답을 "보면서" 각 위치에서 올바른 다음 토큰을 예측하는 법을 배운다. 마치 학생이 모범답안을 보며 "이 상황에서는 이렇게 답해야 한다"를 학습하는 것과 같다. 이것이 현대 LLM 파인튜닝의 핵심 메커니즘이다.

### 1. 인과적 언어 모델링(Causal Language Modeling)

LLM(Llama, GPT 등)은 이전 토큰들을 보고 다음 토큰을 예측하는 방식으로 학습한다. 따라서 학습 데이터에는 프롬프트와 정답이 모두 포함된 전체 문장이 들어가야 한다.

- 학습 원리: 모델은 n번째 토큰까지를 입력으로 받아 n+1번째 토큰을 예측한다.
- 구조: `input_ids`가 `[A, B, C, D]`라면, 모델은 내부적으로 `A`를 보고 `B`를, `A, B`를 보고 `C`를 예측하는 과정을 동시에 수행한다.

### 2. Teacher Forcing 기법

학습 시에는 모델이 이전 단계에서 직접 생성한 토큰을 다음 입력으로 사용하는 것이 아니라, 정답 시퀀스에 있는 실제 이전 토큰을 입력으로 사용한다. 이를 Teacher Forcing이라고 한다.

예를 들어 다음과 같은 시퀀스가 있다고 하자.

~~~text
Position:  [0, 1, 2, 3, 4, 5, 6, 7, 8]
input_ids: [A, B, C, D, E, F, G, H, I]
labels:    [-100, -100, -100, -100, E, F, G, H, I]
~~~

이때 학습 과정은 다음과 같이 이해할 수 있다.

- Position 4: A, B, C, D를 보고 → E 예측
- Position 5: A, B, C, D, E를 보고 → F 예측
- Position 6: A, B, C, D, E, F를 보고 → G 예측

즉, `input_ids`에는 전체 토큰이 들어 있지만, 모델이 특정 위치의 토큰을 예측할 때는 그 위치 이전의 토큰만 참고한다. 뒤쪽 정답 토큰을 미리 보고 맞히는 구조가 아니다.

## 3. Labels와 Loss 계산의 역할

`input_ids`에 정답이 포함되어 있더라도, 모델이 모든 구간에 대해 학습(손실 계산)을 수행하는 것은 아니다. 이때 중요한 역할을 하는 것이 코드에 작성된 `labels`이다.

- `-100`의 의미: PyTorch의 `CrossEntropyLoss`는 기본적으로 레이블 값이 `-100`인 위치를 무시(ignore)한다.
- 학습 차단: 코드에서 프롬프트(User 질문 등) 구간의 레이블을 `-100`으로 설정했기 때문에, 모델이 프롬프트 내용을 예측하며 발생하는 오차는 학습에 반영되지 않는다.
- 학습 집중: 오직 `assistant`의 답변 구간에 해당하는 `labels`만 실제 `input_ids` 값을 가지므로, 모델은 "프롬프트가 주어졌을 때 정답을 생성하는 방법"에 대해서만 가중치를 업데이트한다.

## 학습 vs 추론의 차이

### 학습 시

~~~text
input_ids: <system>당신은 금융분석가</system><user>뉴스내용</user><assistant>분석결과</assistant>
labels:    [-100, -100, ..., -100, 분석결과_토큰들]
~~~

학습 시에는 프롬프트와 정답을 모두 넣는다.  
하지만 `labels`에서 프롬프트 구간은 `-100`으로 처리되기 때문에, 실제 손실 계산은 assistant 답변 구간에서만 일어난다.

### 추론 시

~~~text
input:  <system>당신은 금융분석가</system><user>뉴스내용</user><assistant>
output: 분석결과
~~~

추론 시에는 정답을 넣지 않는다.  
모델은 프롬프트만 입력받고, `<assistant>` 이후부터 분석 결과를 한 토큰씩 생성한다.

### 핵심 정리

- `input_ids`에는 프롬프트와 정답이 모두 들어간다.
- `labels`는 어느 부분을 학습할지 지정하는 역할을 한다.
- `labels`가 `-100`인 구간은 Loss 계산에서 제외된다.
- 따라서 모델은 system/user 프롬프트를 외우는 것이 아니라, assistant 답변 구간을 생성하는 법을 학습한다.
- 학습 시에는 정답을 함께 넣지만, 추론 시에는 정답 없이 프롬프트만 넣고 모델이 답변을 생성한다.